In [0]:
# ==========================================================================
# Databricks — ICP / MarketplaceSimpleAPI — Pipeline de LICENCIAMIENTO
#   subscriptions : SCD Tipo 2 por AccountId (historial de cambios de contrato/SKU/…)
#   audit_log     : insert-only por Id (acumula)
#   preview_invoices: overwrite (foto del ciclo)
#   seat_changes   : MSCSPSeatChanges (historial de Quantity, retroactivo)
#   seat_delta     : MSCSPSeatDelta (asientos inicio/fin, delta, NO asignados)
#   licenciamiento : silver de productos licenciables (SKU/cantidad/asignada/estado/vencimiento)
#   tenants_consumo: silver aparte de Tenant + Consumo Azure (sin asientos)
#   movimientos_licencias: silver de incrementos/reducciones (desde seat_changes)
#
# Auth: GetSessionToken -> "Authenticate: CCPSessionId <token>"
# ==========================================================================

import requests, json, time, datetime as dt
import pandas as pd
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

# --------------------------------------------------------------------------
# Parámetros
# --------------------------------------------------------------------------
dbutils.widgets.text("catalog", "dbx_icp_vnet")
dbutils.widgets.text("reseller_account_id", "281148")
dbutils.widgets.text("audit_start_date", "2018-01-01T00:00:00")
dbutils.widgets.dropdown("pull_audit", "true", ["true", "false"])
dbutils.widgets.dropdown("run_audit_probe", "true", ["true", "false"])
dbutils.widgets.text("audit_probe_company", "459649")

CATALOG           = dbutils.widgets.get("catalog")
CALLER_ACCOUNT_ID = int(dbutils.widgets.get("reseller_account_id"))
AUDIT_START       = dbutils.widgets.get("audit_start_date")
PULL_AUDIT        = dbutils.widgets.get("pull_audit") == "true"
RUN_PROBE         = dbutils.widgets.get("run_audit_probe") == "true"
PROBE_COMP        = dbutils.widgets.get("audit_probe_company")
BRONZE, SILVER    = "icp_bronze", "icp_silver"
BASE_URL = "https://marketplacexpe.intcomexcloud.com/SimpleAPI/SimpleAPIService.svc/rest"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}")

# --------------------------------------------------------------------------
# Autenticación
# --------------------------------------------------------------------------
_session = {"token": None}
def _login():
    u = dbutils.secrets.get("intcomex", "username").strip()
    p = dbutils.secrets.get("intcomex", "password").strip()
    r = requests.post(f"{BASE_URL}/GetSessionToken",
        headers={"Content-Type": "application/json;charset=UTF8", "Accept": "application/json"},
        json={"username": u, "password": p}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"Login ICP falló ({r.status_code}): {r.text[:300]}")
    _session["token"] = r.json()
def _headers():
    if not _session["token"]:
        _login()
    return {"Authenticate": f"CCPSessionId {_session['token']}",
            "Content-Type": "application/json;charset=UTF8", "Accept": "application/json"}
def icp_post(endpoint, payload, _retry=True, _tries=3):
    last = ""
    for attempt in range(_tries):
        r = requests.post(f"{BASE_URL}/{endpoint}", headers=_headers(), json=payload, timeout=180)
        if r.status_code == 200:
            return r.json()
        if _retry and ("IsSessionExpired>true" in r.text or r.status_code == 401):
            _login(); return icp_post(endpoint, payload, _retry=False, _tries=_tries)
        last = f"{endpoint} falló ({r.status_code}): {r.text[:300]}"
        # 5xx transitorios de gateway (502/503/504) -> reintenta con backoff creciente
        if r.status_code in (502, 503, 504) and attempt < _tries - 1:
            time.sleep(2 * (attempt + 1)); continue
        break
    raise RuntimeError(last)

# --------------------------------------------------------------------------
# DIAGNÓSTICO — ¿el reporte de audit devuelve eventos ANTES de 2026?
#   Compara startDate 2018 vs 2026 sobre una empresa con suscripciones viejas.
#   Desactívalo con el widget run_audit_probe = false cuando ya no lo necesites.
# --------------------------------------------------------------------------
if RUN_PROBE:
    print("== Diagnóstico ventana de audit ==")
    print("  AUDIT_START efectivo:", AUDIT_START)
    for sd in ["2018-01-01T00:00:00", "2026-01-01T00:00:00"]:
        try:
            res  = icp_post("ExecuteReport", {"reportName": "Audit Log Object Company",
                   "parameters": {"targetCompany": int(PROBE_COMP), "startDate": sd}})
            cols = [c["Name"] for c in sorted(res.get("Columns", []), key=lambda c: c["CellIndex"])]
            di, ti = cols.index("Date"), cols.index("TargetAccountId")
            rows = res.get("Rows", [])
            dates = [r["Cells"][di] for r in rows]
            print(f"  startDate {sd} -> filas: {len(rows)} | "
                  f"min: {min(dates) if dates else None} | "
                  f"tiene 597703: {any(str(r['Cells'][ti]) == '597703' for r in rows)}")
        except Exception as e:
            print(f"  startDate {sd} -> error: {str(e)[:150]}")
    print("==================================\n")

# --------------------------------------------------------------------------
# Helpers de carga
# --------------------------------------------------------------------------
def _to_spark(pdf):
    if pdf is None or len(pdf) == 0:
        return None
    return spark.createDataFrame(pdf.astype(str).where(pd.notnull(pdf), None))

def save_overwrite(pdf, table):
    df = _to_spark(pdf)
    if df is None:
        print(f"  (sin datos) {table}"); return
    df.withColumn("_ingested_at", F.current_timestamp()).write.format("delta") \
      .mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{BRONZE}.{table}")
    print(f"  OK {table}: {df.count()} filas (overwrite)")

def merge_append(pdf, table, key_cols):
    """Insert-only por key (acumula, no reprocesa)."""
    df = _to_spark(pdf)
    if df is None:
        print(f"  (sin datos) {table}"); return
    df = df.dropDuplicates(key_cols).withColumn("_ingested_at", F.current_timestamp())
    fqn = f"{CATALOG}.{BRONZE}.{table}"
    if not spark.catalog.tableExists(fqn):
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fqn)
        print(f"  creada {table}: {df.count()} filas"); return
    cond = " AND ".join([f"t.`{k}` = s.`{k}`" for k in key_cols])
    (DeltaTable.forName(spark, fqn).alias("t").merge(df.alias("s"), cond)
        .whenNotMatchedInsertAll().execute())
    print(f"  merge {table}: {df.count()} filas evaluadas (insert-only)")

def scd2_merge(pdf, table, key_cols, tracked_cols, bysource_condition=None):
    """SCD Tipo 2: versiona por key. Cambios en tracked_cols -> nueva versión
       (valid_from/valid_to/is_current). Bajas (o no-vienen dentro de scope) -> cierra vigente."""
    df = _to_spark(pdf)
    if df is None:
        print(f"  (sin datos) {table}"); return
    df = df.dropDuplicates(key_cols)
    for c in tracked_cols:                       # asegura que existan las columnas rastreadas
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast("string"))
    df = (df.withColumn("row_hash", F.md5(F.concat_ws("||",
              *[F.coalesce(F.col(c).cast("string"), F.lit("∅")) for c in tracked_cols])))
            .withColumn("last_seen", F.current_timestamp()))
    fqn = f"{CATALOG}.{BRONZE}.{table}"

    if not spark.catalog.tableExists(fqn):
        (df.withColumn("valid_from", F.current_timestamp())
           .withColumn("valid_to", F.lit(None).cast("timestamp"))
           .withColumn("is_current", F.lit(True))
           .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fqn))
        print(f"  creada {table} (SCD2): {df.count()} filas"); return

    tgt = DeltaTable.forName(spark, fqn)
    keycond = " AND ".join([f"t.`{k}` = s.`{k}`" for k in key_cols])
    # Paso 1: cierra versiones vigentes que cambiaron, o que ya no vienen (dentro de scope)
    m = tgt.alias("t").merge(df.alias("s"), f"{keycond} AND t.is_current = true") \
           .whenMatchedUpdate(condition="t.row_hash <> s.row_hash",
                              set={"is_current": "false", "valid_to": "current_timestamp()"})
    bs = "t.is_current = true" + (f" AND ({bysource_condition})" if bysource_condition else "")
    m = m.whenNotMatchedBySourceUpdate(condition=bs,
                                       set={"is_current": "false", "valid_to": "current_timestamp()"})
    m.execute()
    # Paso 2: inserta versiones nuevas (claves nuevas + las que cambiaron)
    current = spark.table(fqn).where("is_current = true").select(*key_cols, "row_hash")
    to_ins = df.join(current, key_cols + ["row_hash"], "left_anti")
    (to_ins.withColumn("valid_from", F.current_timestamp())
           .withColumn("valid_to", F.lit(None).cast("timestamp"))
           .withColumn("is_current", F.lit(True))
           .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(fqn))
    print(f"  SCD2 {table}: +{to_ins.count()} versiones nuevas")

# --------------------------------------------------------------------------
# 1) Empresas (Accounts Modified) + PreviewInvoices (opcional)
#    Accounts Modified NO requiere ONLINEBILL/Read -> de aquí salen IDs y nombres
#    de empresa. GetPreviewInvoices (costos del ciclo) es OPCIONAL: si la cuenta
#    de API no tiene permiso, se avisa y el pipeline continúa.
# --------------------------------------------------------------------------
print("Accounts Modified (empresas)...")
_COMPANY_DISCOVERY_START = "2015-01-01T00:00:00"   # ventana amplia para no perder empresas
am = icp_post("ExecuteReport", {"reportName": "Accounts Modified",
              "parameters": {"startDate": _COMPANY_DISCOVERY_START}})
_ai = {c["Name"]: c["CellIndex"] for c in am.get("Columns", [])}
comp_rows = []
for r in am.get("Rows", []):
    cells = r["Cells"]
    if cells[_ai["AccountType"]] in ("Company", "Reseller"):
        comp_rows.append({"cid": str(cells[_ai["AccountId"]]),
                          "nombre": cells[_ai["DisplayName"]] or cells[_ai.get("ParentCompanyName", _ai["DisplayName"])]})
dim_pd = pd.DataFrame(comp_rows).dropna(subset=["cid"]).drop_duplicates("cid")
save_overwrite(dim_pd, "dim_company")
company_ids = list({CALLER_ACCOUNT_ID,
                    *[int(float(c)) for c in dim_pd["cid"] if str(c).replace('.', '').isdigit()]})
print(f"Empresas a recorrer: {len(company_ids)}")

# PreviewInvoices (costos del ciclo) — requiere permiso ONLINEBILL/Read
try:
    preview = icp_post("GetPreviewInvoices", {"resellerContext": CALLER_ACCOUNT_ID, "groupByDepartments": True})
    inv_rows = []
    for comp in (preview if isinstance(preview, list) else []):
        for ch in comp.get("Charges", []):
            inv_rows.append({"CompanyName": comp.get("CompanyName"), "CompanyAccountId": comp.get("CompanyAccountId"),
                "CompanyVatId": comp.get("CompanyVatId"), "BillingInterval": comp.get("BillingInterval"),
                **{k: ch.get(k) for k in ("ServiceName","ServiceId","AccountId","Costs","CostsOfUnit","SalesPrice",
                    "Currency","UDRCValue","BillableParameter","ActualChargeInterval","VendorName","ProductNumber")}})
    save_overwrite(pd.DataFrame(inv_rows), "preview_invoices")
except Exception as e:
    print(f"  aviso: GetPreviewInvoices no disponible (falta permiso ONLINEBILL/Read?) -> {str(e)[:150]}")

# --------------------------------------------------------------------------
# 2) GetSubscriptions -> aplanar -> SCD2
# --------------------------------------------------------------------------
FIELD_PICK = {"Quantity": "quantity", "MicrosoftTenantId": "tenant_id", "TenantID": "tenant_id",
              "Subscriptionstatus": "subscription_status", "OfferId": "offer_id",
              "BillingType": "billing_type", "Segment": "segment", "SubscriptionName": "subscription_name"}

def flatten_subscription(sub, company_id):
    fields = {f["Name"]: f.get("Value") for f in sub.get("Fields", [])}
    row = {"queried_company_id": company_id, "CompanyAccountId": sub.get("CompanyAccountId"),
        "ParentAccountId": sub.get("ParentAccountId"), "ParentType": sub.get("ParentType"),
        "AccountId": sub.get("AccountId"), "AccountState": sub.get("AccountState"),
        "ServiceName": sub.get("ServiceName"), "ServiceDisplayName": sub.get("ServiceDisplayName"),
        "VendorDisplayName": sub.get("VendorDisplayName"), "BillingStartDate": sub.get("BillingStartDate"),
        "ContractEndDate": sub.get("ContractEndDate"), "ProvisioningStatus": sub.get("ProvisioningStatus"),
        "DependencyAccountId": sub.get("DependencyAccountId"),
        "AdvancePeriodEndAction": sub.get("AdvancePeriodEndAction"),
        "HasRenewActionValuesConfigured": sub.get("HasRenewActionValuesConfigured"),
        "ContractId": sub.get("ContractId"), "PriceProtectionEndDate": sub.get("PriceProtectionEndDate")}
    for src, dst in FIELD_PICK.items():
        if src in fields and (dst not in row or row.get(dst) in (None, "")):
            row[dst] = fields[src]
    monthly = [p for p in sub.get("PriceableItems", []) if p.get("PriceableItemType") == "Monthly"]
    pit = monthly[0] if monthly else (sub.get("PriceableItems") or [{}])[0]
    row["purchase_price"] = pit.get("PurchasePrice"); row["sales_price"] = pit.get("SalesPrice")
    row["price_currency"] = pit.get("Currency"); row["commitment_months"] = pit.get("CommitementPeriodInMonths")
    row["_fields_json"] = json.dumps(fields, ensure_ascii=False, default=str)
    return row

print("Subscriptions...")
sub_rows, ok_ids = [], []
for cid in company_ids:
    try:
        res = icp_post("GetSubscriptions", {"parentAccountId": cid, "excludeUserLevel": False})
        for sub in (res if isinstance(res, list) else []):
            sub_rows.append(flatten_subscription(sub, cid))
        ok_ids.append(str(cid))
    except Exception as e:
        print(f"  aviso: empresa {cid} -> {str(e)[:120]}")

bysrc = None
if ok_ids:
    bysrc = "t.`queried_company_id` IN (" + ",".join(f"'{i}'" for i in ok_ids) + ")"

# atributos cuyo cambio dispara una nueva versión (SKU, contrato, cantidad, estado, precios, fechas, renovación)
TRACKED = ["ContractId", "ServiceName", "ServiceDisplayName", "offer_id", "quantity",
           "subscription_status", "AccountState", "purchase_price", "sales_price",
           "billing_type", "ContractEndDate", "AdvancePeriodEndAction"]
scd2_merge(pd.DataFrame(sub_rows), "subscriptions", ["AccountId"], TRACKED, bysource_condition=bysrc)

# --------------------------------------------------------------------------
# 3) Audit Log -> insert-only por Id
# --------------------------------------------------------------------------
if PULL_AUDIT:
    print("Audit Log...")
    audit_rows = []
    for cid in company_ids:
        try:
            res = icp_post("ExecuteReport", {"reportName": "Audit Log Object Company",
                           "parameters": {"targetCompany": cid, "startDate": AUDIT_START}})
            cols = sorted(res.get("Columns", []), key=lambda c: c["CellIndex"])
            names = [c["Name"] for c in cols]
            for r in res.get("Rows", []):
                d = dict(zip(names, r["Cells"])); d["queried_company_id"] = cid
                audit_rows.append(d)
        except Exception as e:
            print(f"  aviso: audit {cid} -> {str(e)[:120]}")
    merge_append(pd.DataFrame(audit_rows), "audit_log", ["Id"])

# --------------------------------------------------------------------------
# 3.5) Reportes nativos de asientos (retroactivos, vía ExecuteReport)
#   seat_changes : historial de valores de Quantity con ventana de vigencia
#                  (ValueStartDate/ValueEndDate) -> incrementos/reducciones
#   seat_delta   : por suscripción, SeatsAtStart/SeatsAtEnd/SeatDelta y
#                  UnassignedLicenses -> permite calcular la cantidad ASIGNADA
#                  (SeatsAtEnd - UnassignedLicenses) sin Microsoft Graph
# --------------------------------------------------------------------------
SEATRPT_START = "2022-01-01T00:00:00"
SEATRPT_END   = "2035-12-31T23:59:59"   # ventana amplia: incluye intervalos abiertos
print("Seat reports (MSCSPSeatChanges / MSCSPSeatDelta)...")
for _rep, _tab in [("MSCSPSeatChanges", "seat_changes"), ("MSCSPSeatDelta", "seat_delta")]:
    try:
        r = icp_post("ExecuteReport", {"reportName": _rep,
              "parameters": {"startDate": SEATRPT_START, "endDate": SEATRPT_END}})
        _c = [c["Name"] for c in sorted(r.get("Columns", []), key=lambda c: c["CellIndex"])]
        _rows = [dict(zip(_c, row["Cells"])) for row in r.get("Rows", [])]
        save_overwrite(pd.DataFrame(_rows), _tab)
    except Exception as e:
        print(f"  aviso: {_rep} -> {str(e)[:150]}")

# --------------------------------------------------------------------------
# 4) Silver — versión VIGENTE + nombre empresa + asignadas + trazabilidad
# --------------------------------------------------------------------------
# subs: última versión conocida de cada suscripción = vigente del feed (is_current=true)
#       + retiradas (última versión ya no vigente, es decir salieron del feed / dadas de baja).
_ball  = spark.table(f"{CATALOG}.{BRONZE}.subscriptions")
_wlast = Window.partitionBy("AccountId").orderBy(F.col("valid_from").desc())
subs   = (_ball.withColumn("_rn", F.row_number().over(_wlast)).filter("_rn = 1").drop("_rn")
               .withColumn("vigente_en_feed",
                           F.when(F.col("is_current") == True, F.lit("Sí")).otherwise(F.lit("No"))))
audit = spark.table(f"{CATALOG}.{BRONZE}.audit_log")

ev = (audit.withColumn("_d", F.to_timestamp("Date"))
           .withColumn("_is_create", F.when(F.lower(F.col("Description")).like("%has been created%"), 1).otherwise(0)))
# ACTIVADOR = usuario del evento de CREACIÓN (o el más antiguo si no hay 'created') -> el humano que dio de alta
_wa = Window.partitionBy("TargetAccountId").orderBy(F.col("_is_create").desc(), F.col("_d").asc())
activador = (ev.withColumn("_rn", F.row_number().over(_wa)).filter("_rn = 1")
    .select(F.col("TargetAccountId").cast("string").alias("_acct_a"),
            F.col("UserUsername").alias("activado_por"),
            F.col("_d").alias("fecha_activacion")))
# ÚLTIMO CAMBIO = evento más reciente (útil para auditoría de modificaciones)
_wl = Window.partitionBy("TargetAccountId").orderBy(F.col("_d").desc())
ultimo = (ev.withColumn("_rn", F.row_number().over(_wl)).filter("_rn = 1")
    .select(F.col("TargetAccountId").cast("string").alias("_acct_u"),
            F.col("UserUsername").alias("ultimo_cambio_por"),
            F.col("EventType").alias("ultimo_evento"),
            F.col("_d").alias("fecha_ultimo_cambio")))

# dimensión de nombres: dim_company (Accounts Modified, prioridad 1) +
# preview_invoices si existe (2) + eventos de empresa del audit (3)
dims = [spark.table(f"{CATALOG}.{BRONZE}.dim_company")
            .select(F.col("cid").cast("string").alias("cid"), F.col("nombre"))
            .where("nombre is not null").withColumn("_pri", F.lit(1))]
if spark.catalog.tableExists(f"{CATALOG}.{BRONZE}.preview_invoices"):
    dims.append(spark.table(f"{CATALOG}.{BRONZE}.preview_invoices")
        .select(F.col("CompanyAccountId").cast("string").alias("cid"), F.col("CompanyName").alias("nombre"))
        .where("nombre is not null").withColumn("_pri", F.lit(2)))
dims.append(audit.where("TargetAccountType = 'Company'")
    .select(F.col("TargetAccountId").cast("string").alias("cid"), F.col("TargetDisplayName").alias("nombre"))
    .where("nombre is not null").withColumn("_pri", F.lit(3)))
dim_all = dims[0]
for d in dims[1:]:
    dim_all = dim_all.unionByName(d)
_wc = Window.partitionBy("cid").orderBy("_pri")
dim_company = dim_all.withColumn("_rn", F.row_number().over(_wc)).filter("_rn = 1").select("cid", "nombre")

base = subs.select(
    F.col("CompanyAccountId").alias("empresa_account_id"),
    F.col("tenant_id"),
    F.col("ServiceName").alias("sku_id"),
    F.col("ServiceDisplayName").alias("sku_nombre"),
    F.col("VendorDisplayName").alias("vendor"),
    F.expr("cast(try_cast(quantity as double) as int)").alias("cantidad_contratada"),
    F.coalesce(F.col("subscription_status"), F.col("AccountState")).alias("estado"),
    F.to_timestamp("BillingStartDate").alias("fecha_alta"),
    F.to_timestamp("ContractEndDate").alias("fecha_fin_contrato"),
    F.col("AdvancePeriodEndAction").alias("accion_fin_periodo"),
    F.when(F.col("AdvancePeriodEndAction").isNull(), None)
     .otherwise(F.col("AdvancePeriodEndAction") != F.lit("Terminate")).alias("es_renovable"),
    (F.lower(F.coalesce(F.col("billing_type"), F.lit(""))).contains("commitment")
     | (F.expr("try_cast(commitment_months as int)") > 0)).alias("tiene_compromiso"),
    F.expr("try_cast(commitment_months as int)").alias("meses_compromiso"),
    F.col("billing_type").alias("tipo_facturacion"),
    F.to_timestamp("PriceProtectionEndDate").alias("fin_proteccion_precio"),
    F.col("ContractId").alias("contrato_id"),
    F.expr("try_cast(purchase_price as double)").alias("costo_unit_compra"),
    F.expr("try_cast(sales_price as double)").alias("precio_unit_venta"),
    F.col("price_currency").alias("moneda"),
    F.col("AccountId").cast("string").alias("subscription_id"),
    F.to_timestamp("valid_from").alias("version_desde"),
    F.col("vigente_en_feed"),
)

silver = (base
    .join(activador, base.subscription_id == activador._acct_a, "left").drop("_acct_a")
    .join(ultimo,    base.subscription_id == ultimo._acct_u,   "left").drop("_acct_u")
    .join(dim_company, base.empresa_account_id == dim_company.cid, "left").drop("cid")
    .withColumn("empresa_nombre", F.coalesce(F.col("nombre"), F.col("empresa_account_id"))).drop("nombre"))

# cantidad efectivamente ASIGNADA A USUARIOS = Microsoft Graph (consumedUnits).
#   NOTA: MSCSPSeatDelta.UnassignedLicenses viene 0 (mide aprovisionamiento CSP,
#   no asignación real en el tenant), así que NO sirve como asignada -> solo Graph.
_gfqn = f"{CATALOG}.{BRONZE}.graph_assigned"
if spark.catalog.tableExists(_gfqn):
    g = spark.table(_gfqn).select(F.col("tenant_id").alias("_gt"), F.col("match_key").alias("_gk"),
                                  F.col("cantidad_asignada").cast("int").alias("cantidad_asignada"))
    silver = silver.join(g, (F.col("tenant_id") == F.col("_gt")) &
                            (F.col("sku_nombre").contains(F.col("_gk"))), "left").drop("_gt", "_gk")
else:
    silver = silver.withColumn("cantidad_asignada", F.lit(None).cast("int"))

# ---- limpieza para Power BI: categoría + etiquetas legibles (menos nulos) ----
silver = (silver
    .withColumn("categoria",
        F.when(F.col("sku_nombre").contains("Organization tenant"), F.lit("Tenant"))
         .when(F.col("sku_nombre").contains("Azure Plan"), F.lit("Consumo Azure"))
         .when(F.col("sku_nombre").rlike("(?i)perpetual"), F.lit("Perpetua"))
         .when(F.col("sku_nombre").rlike("(?i)sophos|aws|migrationwiz|migration bundle|report sending|indirect reseller|distribution enablement"), F.lit("Servicio"))
         .otherwise(F.lit("Licencia")))
    # booleanos -> etiqueta legible (evita '(Blank)' en slicers)
    .withColumn("renovacion", F.when(F.col("es_renovable") == True, F.lit("Renovable"))
                               .when(F.col("es_renovable") == False, F.lit("No renovable"))
                               .otherwise(F.lit("N/A")))
    .withColumn("compromiso", F.when(F.col("tiene_compromiso") == True, F.lit("Con compromiso"))
                               .otherwise(F.lit("Sin compromiso")))
    # dimensiones de texto: nulo -> etiqueta
    .withColumn("tenant_id",        F.coalesce("tenant_id", F.lit("N/A")))
    .withColumn("contrato_id",      F.coalesce("contrato_id", F.lit("N/A")))
    .withColumn("tipo_facturacion", F.coalesce("tipo_facturacion", F.lit("N/A")))
    .withColumn("moneda",           F.coalesce("moneda", F.lit("N/A")))
    .withColumn("activado_por",     F.coalesce("activado_por", F.lit("Sin registro")))
    .withColumn("ultimo_cambio_por",F.coalesce("ultimo_cambio_por", F.lit("Sin registro")))
    .withColumn("ultimo_evento",    F.coalesce("ultimo_evento", F.lit("Sin registro")))
    # estado normalizado (Activo / Suspendido / Cancelado / Deshabilitado)
    .withColumn("estado",
        F.when(F.lower(F.col("estado")).contains("active"),       F.lit("Activo"))
         .when(F.lower(F.col("estado")).contains("suspend"),      F.lit("Suspendido"))
         .when(F.lower(F.col("estado")).rlike("cancel|terminat"), F.lit("Cancelado"))
         .when(F.lower(F.col("estado")).contains("disab"),        F.lit("Deshabilitado"))
         .otherwise(F.col("estado")))
    # retiradas del feed que quedaron con estado "Activo" -> marcarlas como Retirado
    # (ICP no cambia el status al dar de baja; solo deja de devolver la suscripción)
    .withColumn("estado",
        F.when((F.col("vigente_en_feed") == "No") & (F.col("estado") == "Activo"), F.lit("Retirado"))
         .otherwise(F.col("estado")))
    # fecha fin "real": ignora centinelas lejanísimas (año >= 2100 = sin vencimiento real,
    # p.ej. Azure Plan / consumo) para no mostrar días/meses gigantescos
    .withColumn("_fin_real",
        F.when(F.col("fecha_fin_contrato").isNotNull() &
               (F.year(F.to_date(F.col("fecha_fin_contrato"))) < 2100),
               F.to_date(F.col("fecha_fin_contrato"))))
    # días y meses para vencer (nulo si no tiene vencimiento real)
    .withColumn("dias_para_vencer",
        F.when(F.col("_fin_real").isNotNull(), F.datediff(F.col("_fin_real"), F.current_date())))
    .withColumn("meses_para_vencer",
        F.when(F.col("_fin_real").isNotNull(),
               F.round(F.months_between(F.col("_fin_real"), F.current_date()), 1)))
    # próximos a vencer: contrato que vence dentro de 5 días (>=0 y <=5)
    .withColumn("por_vencer",
        F.when((F.col("dias_para_vencer") >= 0) & (F.col("dias_para_vencer") <= 5), F.lit("Sí"))
         .otherwise(F.lit("No")))
    # etiqueta de vencimiento (evita nulos confusos en Power BI)
    .withColumn("vencimiento",
        F.when(F.col("_fin_real").isNull(),   F.lit("Sin vencimiento"))
         .when(F.col("dias_para_vencer") < 0, F.lit("Vencido"))
         .otherwise(F.lit("Vigente")))
    # proyecto asociado: ContractId de ICP = código de proyecto/orden (p.ej. 2411DP001)
    .withColumn("proyecto", F.col("contrato_id"))
    # estado unificado con el vocabulario del requerimiento (una sola columna para el slicer 'Estado')
    .withColumn("estado_licencia",
        F.when(F.col("vigente_en_feed") == "No",   F.lit("Retirado"))
         .when(F.col("estado") == "Suspendido",     F.lit("Suspendido"))
         .when(F.col("estado") == "Cancelado",      F.lit("Cancelado"))
         .when(F.col("estado") == "Deshabilitado",  F.lit("Deshabilitado"))
         .when(F.col("vencimiento") == "Vencido",   F.lit("Expirado"))
         .when(F.col("por_vencer") == "Sí",         F.lit("Próximo a vencer"))
         .otherwise(F.lit("Activo")))
    # brecha de activación: contratadas - activadas (alerta operativa del requerimiento)
    .withColumn("licencias_sin_asignar",
        F.when(F.col("cantidad_asignada").isNotNull(),
               F.greatest(F.col("cantidad_contratada") - F.col("cantidad_asignada"), F.lit(0))))
    .drop("es_renovable", "tiene_compromiso", "accion_fin_periodo", "meses_compromiso", "_fin_real",
          "contrato_id"))   # contrato_id == proyecto (mismo ContractId): se deja solo 'proyecto'

_first = ["categoria", "empresa_account_id", "empresa_nombre", "tenant_id",
          "proyecto", "sku_id", "sku_nombre",
          "cantidad_contratada", "cantidad_asignada", "licencias_sin_asignar",
          "estado_licencia", "estado", "vigente_en_feed", "vencimiento", "por_vencer",
          "dias_para_vencer", "meses_para_vencer", "renovacion", "compromiso"]
silver = silver.select(*[c for c in _first if c in silver.columns],
                       *[c for c in silver.columns if c not in _first])

# --------------------------------------------------------------------------
# Separación para el dashboard:
#   licenciamiento   -> productos licenciables (con SKU, cantidad y modalidad)
#   tenants_consumo  -> registros de Tenant y Consumo Azure (sin asientos; traen
#                       nulos en cantidad/modalidad/vencimiento y ensuciarían los
#                       totales del dashboard). Se guardan aparte para trazabilidad.
# --------------------------------------------------------------------------
CATEG_LICENCIABLE = ["Licencia", "Perpetua", "Servicio"]
lic   = silver.where(F.col("categoria").isin(CATEG_LICENCIABLE))
infra = silver.where(~F.col("categoria").isin(CATEG_LICENCIABLE))

(lic.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(f"{CATALOG}.{SILVER}.licenciamiento"))
(infra.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.{SILVER}.tenants_consumo"))

print(f"\nSilver licenciamiento (productos licenciables): {lic.count()} filas")
print(f"Silver tenants_consumo (Tenant + Consumo Azure): {infra.count()} filas")
print("\n  licenciamiento — desglose por estado_licencia:")
lic.groupBy("estado_licencia").count().orderBy(F.col("count").desc()).show(20, False)
print("  categoria (universo completo antes de separar):")
silver.groupBy("categoria").count().orderBy(F.col("count").desc()).show(20, False)
display(lic.limit(30))

# --------------------------------------------------------------------------
# 5) Movimientos de licencias (incrementos / reducciones) — desde MSCSPSeatChanges
#   Retroactivo: cada cambio de Quantity queda con su valor anterior, valor nuevo,
#   delta, tipo y fecha exacta del cambio. No depende de acumular corridas.
# --------------------------------------------------------------------------
_scqn = f"{CATALOG}.{BRONZE}.seat_changes"
if spark.catalog.tableExists(_scqn):
    sc = (spark.table(_scqn).where("FieldName = 'Quantity'")
          .withColumn("valor", F.expr("try_cast(Value as double)").cast("int"))
          .withColumn("desde", F.to_timestamp("ValueStartDate")))
    _wm = Window.partitionBy("ServiceAccountId").orderBy("desde")
    mov = (sc.withColumn("valor_prev", F.lag("valor").over(_wm))
             .where("valor_prev is not null and valor <> valor_prev")
             .withColumn("delta", F.col("valor") - F.col("valor_prev"))
             .withColumn("tipo", F.when(F.col("delta") > 0, F.lit("Incremento")).otherwise(F.lit("Reducción")))
             .select(F.col("ServiceAccountId").cast("string").alias("subscription_id"),
                     F.col("CompanyName").alias("empresa_nombre"),
                     F.col("ServiceName").alias("sku_nombre"),
                     F.col("valor_prev").alias("cantidad_antes"),
                     F.col("valor").alias("cantidad_despues"),
                     "delta", "tipo",
                     F.col("desde").alias("fecha_cambio"))
             .orderBy(F.col("desde").desc()))
    (mov.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SILVER}.movimientos_licencias"))
    print(f"\nMovimientos de licencias (incrementos/reducciones): {mov.count()} filas")
    mov.groupBy("tipo").agg(F.count('*').alias('n'), F.sum('delta').alias('delta_total')).show()
else:
    print("\n  (sin seat_changes: no se generó movimientos_licencias)")

In [0]:
lic = spark.table("dbx_icp_vnet.icp_silver.licenciamiento")
print("columnas:", len(lic.columns))
print("con cantidad_asignada:", lic.where("cantidad_asignada is not null").count(), "de", lic.count())

In [0]:
sd = spark.table("dbx_icp_vnet.icp_bronze.seat_delta")
sd.select("CompanyName","ProductDisplayName","SeatsAtStart","SeatsAtEnd",
          "SeatDelta","UnassignedLicenses").show(20, False)
print("distribución de UnassignedLicenses:")
sd.selectExpr("try_cast(UnassignedLicenses as double) u").groupBy("u").count().orderBy("u").show()

In [0]:
from pyspark.sql import functions as F

# Patrón válido: 4 dígitos (AAMM) + 2 letras + 3 dígitos, con sufijo opcional -XX
PATRON_OK = r'^[0-9]{4}[A-Z]{2}[0-9]{3}(-[A-Z0-9]+)?$'

lic = spark.table("dbx_icp_vnet.icp_silver.licenciamiento")

aud = lic.withColumn("calidad_proyecto",
    F.when(F.col("proyecto").rlike(PATRON_OK), F.lit("OK"))
     # Licencia NCE sin código = hallazgo (debería tener proyecto)
     .when((F.col("proyecto") == "N/A") & (F.col("categoria") == "Licencia"),
           F.lit("Falta proyecto (Licencia sin código)"))
     # Perpetua/Servicio sin código = esperado (no siempre llevan proyecto)
     .when(F.col("proyecto") == "N/A", F.lit("Sin proyecto (esperado)"))
     # Formato inválido: basura, espacios, varios códigos juntos, etc.
     .otherwise(F.lit("No estándar / basura")))

print("=== Resumen de calidad del campo proyecto ===")
aud.groupBy("calidad_proyecto").count().orderBy(F.col("count").desc()).show(10, False)

# ---- Reporte de auditoría: solo los hallazgos ----
hallazgos = (aud
    .where("calidad_proyecto IN ('No estándar / basura', 'Falta proyecto (Licencia sin código)')")
    .select("calidad_proyecto", "empresa_account_id", "empresa_nombre", "proyecto",
            "categoria", "sku_nombre", "cantidad_contratada", "estado_licencia",
            "subscription_id", "activado_por", "fecha_alta", "fecha_activacion")
    .orderBy("calidad_proyecto", "empresa_nombre", "proyecto"))

print(f"\nHallazgos para auditoría: {hallazgos.count()} filas")
display(hallazgos)   # desde este grid puedes descargar a CSV/Excel

In [0]:
from pyspark.sql import functions as F

PATRON_OK = r'^[0-9]{4}[A-Z]{2}[0-9]{3}(-[A-Z0-9]+)?$'

# Universo COMPLETO: licenciables + tenants/consumo (la basura puede estar en cualquiera)
lic      = spark.table("dbx_icp_vnet.icp_silver.licenciamiento")
infra    = spark.table("dbx_icp_vnet.icp_silver.tenants_consumo")
universo = lic.unionByName(infra, allowMissingColumns=True)

aud = universo.withColumn("calidad_proyecto",
    F.when(F.col("proyecto").rlike(PATRON_OK), F.lit("OK"))
     .when((F.col("proyecto") == "N/A") & (F.col("categoria") == "Licencia"),
           F.lit("Falta proyecto (Licencia sin código)"))
     .when(F.col("proyecto") == "N/A", F.lit("Sin proyecto (esperado)"))
     .otherwise(F.lit("No estándar / basura")))

print("=== Resumen (universo completo: licenciamiento + tenants_consumo) ===")
aud.groupBy("calidad_proyecto").count().orderBy(F.col("count").desc()).show(10, False)

hallazgos = (aud
    .where("calidad_proyecto IN ('No estándar / basura', 'Falta proyecto (Licencia sin código)')")
    .select("calidad_proyecto","categoria","empresa_account_id","empresa_nombre","proyecto",
            "sku_nombre","cantidad_contratada","estado_licencia","subscription_id",
            "activado_por","fecha_alta","fecha_activacion")
    .orderBy("calidad_proyecto","empresa_nombre","proyecto"))
print(f"\nHallazgos: {hallazgos.count()} filas")
display(hallazgos)   # descargable a CSV/Excel desde el grid

In [0]:
df = spark.table("dbx_icp_vnet.icp_silver.licenciamiento")
print([c for c in df.columns if c in ("estado","por_vencer","dias_para_vencer")])
display(df.select("empresa_nombre","sku_nombre","estado","por_vencer","dias_para_vencer","activado_por")
          .where("por_vencer = 'Sí'"))

In [0]:
%sql
SELECT COUNT(*)                       AS total,
       MIN(to_timestamp(Date))        AS mas_antiguo,
       MAX(to_timestamp(Date))        AS mas_reciente
FROM dbx_icp_vnet.icp_bronze.audit_log;

In [0]:
%sql
SELECT TargetAccountId, UserUsername, EventCategory, EventType,
       to_timestamp(Date) AS fecha, Description
FROM dbx_icp_vnet.icp_bronze.audit_log
WHERE TargetAccountId = 597703
ORDER BY fecha;

In [0]:
import pandas as pd
cid = 467136
res = icp_post("ExecuteReport", {"reportName": "Audit Log Object Company",
               "parameters": {"targetCompany": cid, "startDate": AUDIT_START}})
cols = sorted(res.get("Columns", []), key=lambda c: c["CellIndex"])
names = [c["Name"] for c in cols]
rows = []
for r in res.get("Rows", []):
    d = dict(zip(names, r["Cells"])); d["queried_company_id"] = cid
    rows.append(d)
merge_append(pd.DataFrame(rows), "audit_log", ["Id"])
print(f"467136 -> {len(rows)} filas de audit mergeadas")

In [0]:
from pyspark.sql import functions as F
DESDE = "2026-09-01"   # ajusta la ventana

uni = (spark.table("dbx_icp_vnet.icp_silver.licenciamiento")
        .unionByName(spark.table("dbx_icp_vnet.icp_silver.tenants_consumo"), allowMissingColumns=True))

nuevas = (uni.where(F.to_date("fecha_alta") >= F.lit(DESDE))
    .select("empresa_nombre","proyecto","sku_nombre","cantidad_contratada",
            "estado_licencia","fecha_alta","activado_por")
    .orderBy(F.col("fecha_alta").desc()))
print(f"Altas nuevas desde {DESDE}: {nuevas.count()}")
display(nuevas)

In [0]:
from pyspark.sql import functions as F, Window as W
b = spark.table("dbx_icp_vnet.icp_bronze.subscriptions") \
         .withColumn("q", F.expr("cast(try_cast(quantity as double) as int)"))
w = W.partitionBy("AccountId").orderBy("valid_from")

cambios = (b
    .withColumn("q_prev", F.lag("q").over(w))
    .withColumn("fecha_prev", F.lag("valid_from").over(w))
    .where("q_prev is not null and q <> q_prev")          # solo cuando cambió la cantidad
    .withColumn("delta", F.col("q") - F.col("q_prev"))
    .withColumn("tipo", F.when(F.col("delta") > 0, "Incremento").otherwise("Reducción"))
    .select(F.col("CompanyAccountId").alias("empresa"), "ServiceDisplayName",
            F.col("ContractId").alias("proyecto"),
            F.col("q_prev").alias("cant_antes"), F.col("q").alias("cant_despues"),
            "delta","tipo", F.col("fecha_prev").alias("version_anterior"),
            F.col("valid_from").alias("version_nueva"))
    .orderBy(F.col("valid_from").desc()))

print(f"Cambios de cantidad: {cambios.count()}")
cambios.groupBy("tipo").agg(F.count("*").alias("n"), F.sum("delta").alias("delta_total")).show()
display(cambios)

In [0]:
for rep in ["MSCSPSeatChanges", "MSCSPSeatDelta"]:
    try:
        res = icp_post("ExecuteReport", {"reportName": rep,
              "parameters": {"startDate": "2022-01-01T00:00:00",
                             "endDate": "2026-12-31T23:59:59"}})
        cols = [c["Name"] for c in sorted(res.get("Columns", []), key=lambda c: c["CellIndex"])]
        print(f"\n{rep}: {len(res.get('Rows', []))} filas\n  columnas: {cols}")
        if res.get("Rows"): print("  ejemplo:", res["Rows"][0]["Cells"])
    except Exception as e:
        print(f"\n{rep}: error -> {str(e)[:200]}")